In [25]:
import requests
from bs4 import BeautifulSoup
import json
import os

URL = "https://en.wikipedia.org/wiki/Lionel_Messi"

headers = {
    "User-Agent": "Aula-CPA"
}

#Criar pasta automaticamente
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "messi_links.json")

#Garantir sobrescrita total (opcional, mas seguro)
if os.path.exists(output_file):
    os.remove(output_file)

#Requisição
response = requests.get(URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Sempre reinicia a estrutura
links = set()

for a in soup.find_all("a", href=True):
    href = a["href"]

    if href.startswith("/wiki/") and ":" not in href:
        full_link = "https://en.wikipedia.org" + href
        links.add(full_link)

# Limitar quantidade (e evitar duplicatas)
links = list(links)

#Salvar (sempre sobrescreve)
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(links, f, indent=4, ensure_ascii=False)

print(f"{len(links)} links coletados.")
print(f"Arquivo salvo em: {output_file}")

2051 links coletados.
Arquivo salvo em: data\messi_links.json


In [26]:
import time

html_dir = os.path.join(output_dir, "html")
os.makedirs(html_dir, exist_ok=True)

#Limpar apenas os arquivos dentro da pasta
for file in os.listdir(html_dir):
    file_path = os.path.join(html_dir, file)
    
    if os.path.isfile(file_path):
        os.remove(file_path)

#garantir consistência
links = list(links)

# salvar página principal
with open(os.path.join(html_dir, "main.html"), "w", encoding="utf-8") as f:
    f.write(response.text)

# baixar páginas dos links
for i, link in enumerate(links):
    try:
        r = requests.get(link, headers=headers)

        file_path = os.path.join(html_dir, f"page_{i}.html")

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(r.text)

        time.sleep(1)

    except Exception:
        continue

In [ ]:
from datetime import datetime
import pandas as pd

#Sempre reinicia (evita duplicação)
data_pages = []
data_images = []

for file in os.listdir(html_dir):
    path = os.path.join(html_dir, file)

    if not os.path.isfile(path):
        continue

    with open(path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    #timestamp
    timestamp = datetime.now()

    #título
    title_tag = soup.find("h1")
    title = title_tag.text if title_tag else None

    #conteúdo principal
    content = soup.find("div", {"id": "mw-content-text"})
    first_paragraph = None

    if content:
        paragraphs = content.find_all("p", recursive=True)

        for p in paragraphs:
            text = p.get_text(strip=True)
            if text:
                first_paragraph = text
                break

    #imagens
    images = [
        img["src"] for img in soup.find_all("img")
        if img.get("src")
    ]

    #links internos
    internal_links = [
        a["href"] for a in soup.find_all("a", href=True)
        if a["href"].startswith("/wiki/")
    ]

    #salva uma vez só
    data_pages.append({
        "title": title,
        "first_paragraph": first_paragraph,
        "internal_links": len(internal_links),  # melhor métrica
        "timestamp": timestamp
    })

    #imagens
    for img in images:
        data_images.append({
            "page": title,
            "image_url": img,
            "timestamp": timestamp
        })

In [12]:
df_pages = pd.DataFrame(data_pages)
df_images = pd.DataFrame(data_images)

df_pages.to_csv("data/pages.csv",sep=";", index=False)
df_images.to_csv("data/images.csv", sep=";", index=False)

In [17]:
df_pages.tail(20)

,title,first_paragraph,internal_links,timestamp
42,The Sporting News,The Sporting Newsis a website and former magaz...,/wiki/Main_Page | /wiki/Wikipedia:Contents | /...,2026-04-06 15:22:28.356348
43,The Sporting News,The Sporting Newsis a website and former magaz...,250,2026-04-06 15:22:28.423567
44,YouTube,YouTubeis an Americanonline video sharingplatf...,/wiki/Main_Page | /wiki/Wikipedia:Contents | /...,2026-04-06 15:22:28.423567
45,YouTube,YouTubeis an Americanonline video sharingplatf...,2167,2026-04-06 15:22:29.135871
46,Lee Trevino,"Lee Buck Trevino(born December 1, 1939) is an ...",/wiki/Main_Page | /wiki/Wikipedia:Contents | /...,2026-04-06 15:22:29.135871
47,Lee Trevino,"Lee Buck Trevino(born December 1, 1939) is an ...",1968,2026-04-06 15:22:29.687373
48,Gabriel Moya (footballer),Gabriel Moya Sanz(born 20 March 1966) is a Spa...,/wiki/Main_Page | /wiki/Wikipedia:Contents | /...,2026-04-06 15:22:29.687373
49,Gabriel Moya (footballer),Gabriel Moya Sanz(born 20 March 1966) is a Spa...,81,2026-04-06 15:22:29.720691
50,2013 FIFA Club World Cup,The2013 FIFA Club World Cup(officially known a...,/wiki/Main_Page | /wiki/Wikipedia:Contents | /...,2026-04-06 15:22:29.720691
51,2013 FIFA Club World Cup,The2013 FIFA Club World Cup(officially known a...,576,2026-04-06 15:22:29.840062
